Overall Accuracy - SVAMP

Normal:
CoT - 85.85
Standard - 82.93
Complex CoT - 85.37

Hypothesis:
CoT - 86.83
Standard - 89.27
Complex CoT - 85.37

In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [3]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0,
        model=deployment
    )

In [6]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/ASDIVsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/h_CoT_bad.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

def process_entry(d):
    """Process a single entry from dev_data."""
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then think step by step through this plan. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step, and correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect" or result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:01<06:06,  1.80s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:03<02:28,  1.36it/s]

Accuracy: 3 / 4 = 75.00%
Accuracy: 4 / 5 = 80.00%
Accuracy: 5 / 6 = 83.33%
Accuracy: 6 / 7 = 85.71%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 11 / 12 = 91.67%
Accuracy: 12 / 13 = 92.31%
Accuracy: 12 / 14 = 85.71%
Accuracy: 13 / 15 = 86.67%
Accuracy: 14 / 16 = 87.50%
Accuracy: 15 / 17 = 88.24%
Accuracy: 16 / 18 = 88.89%
Accuracy: 17 / 19 = 89.47%
Accuracy: 18 / 20 = 90.00%
Accuracy: 19 / 21 = 90.48%
Accuracy: 20 / 22 = 90.91%


 11%|█         | 23/205 [00:03<00:21,  8.44it/s]

...

Accuracy: 172 / 197 = 87.31%
Accuracy: 172 / 198 = 86.87%
Accuracy: 173 / 199 = 86.93%
Accuracy: 174 / 200 = 87.00%
Accuracy: 175 / 201 = 87.06%
Accuracy: 176 / 202 = 87.13%


 99%|█████████▉| 203/205 [02:07<00:00,  2.85it/s]

Accuracy: 177 / 203 = 87.19%


100%|██████████| 205/205 [02:09<00:00,  1.59it/s]

Accuracy: 177 / 204 = 86.76%
Accuracy: 178 / 205 = 86.83%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/h_Standard_bad.txt'

# === Cleaning Utility ===
def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then answer. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  1%|          | 2/205 [00:00<01:04,  3.16it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:00<00:57,  3.51it/s]

Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%


  4%|▍         | 9/205 [00:01<00:24,  7.85it/s]

...

Accuracy: 172 / 189 = 91.01%
Accuracy: 173 / 190 = 91.05%
Accuracy: 174 / 191 = 91.10%
Accuracy: 175 / 192 = 91.15%
Accuracy: 175 / 193 = 90.67%
Accuracy: 176 / 194 = 90.72%
Accuracy: 176 / 195 = 90.26%


 96%|█████████▌| 196/205 [02:54<00:09,  1.10s/it]

Accuracy: 177 / 196 = 90.31%


100%|██████████| 205/205 [02:55<00:00,  1.17it/s]

Accuracy: 178 / 197 = 90.36%
Accuracy: 178 / 198 = 89.90%
Accuracy: 178 / 199 = 89.45%
Accuracy: 179 / 200 = 89.50%
Accuracy: 180 / 201 = 89.55%
Accuracy: 181 / 202 = 89.60%
Accuracy: 182 / 203 = 89.66%
Accuracy: 182 / 204 = 89.22%
Accuracy: 183 / 205 = 89.27%


91.50 + 2/100 = 92.50, check hypothesis_Standerd

In [8]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/ASDIV/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['body'] + ' ' + d['question']
        match = re.search(r'(\d+\.?\d*)', d['answer'])  # Extract ground truth number
        if match:
            a = float(match.group(1))
        else:
            raise ValueError(f"No numeric value found in answer: {d['answer']}")

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Finish your response with: the answer is <answer>."
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: the answer is <answer>."
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Call ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Numerical Answer ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block ===
        log_block = (
            f'Q: {q}\n'
            f'RESPONSE:\n{ans_model}\n'
            f'EXTRACTED:\n{extracted}\n'
            f'GROUND_TRUTH:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception:
        err_log = f"⚠️ Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", err_log

# === Main Parallel Execution ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                if result_type == "error":
                    error_count += 1
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)


  0%|          | 1/205 [00:02<09:08,  2.69s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%


  2%|▏         | 5/205 [00:03<01:50,  1.82it/s]

Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 9 / 9 = 100.00%
Accuracy: 10 / 10 = 100.00%
Accuracy: 11 / 11 = 100.00%
Accuracy: 12 / 12 = 100.00%


  6%|▋         | 13/205 [00:04<00:41,  4.66it/s]

...

Accuracy: 162 / 189 = 85.71%
Accuracy: 163 / 190 = 85.79%
Accuracy: 164 / 191 = 85.86%
Accuracy: 165 / 192 = 85.94%
Accuracy: 166 / 193 = 86.01%
Accuracy: 167 / 194 = 86.08%
Accuracy: 167 / 195 = 85.64%


 96%|█████████▌| 196/205 [04:14<00:06,  1.40it/s]

Accuracy: 168 / 196 = 85.71%
Accuracy: 169 / 197 = 85.79%
Accuracy: 169 / 198 = 85.35%
Accuracy: 170 / 199 = 85.43%


100%|██████████| 205/205 [04:24<00:00,  1.29s/it]

Accuracy: 171 / 200 = 85.50%
Accuracy: 172 / 201 = 85.57%
Accuracy: 173 / 202 = 85.64%
Accuracy: 174 / 203 = 85.71%
Accuracy: 174 / 204 = 85.29%
Accuracy: 175 / 205 = 85.37%

✅ Accuracy: 175 / 205 = 85.37%
❌ Errors: 1

